In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import dlib
import cv2 as cv
import matplotlib.pyplot as plt

In [2]:
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("./model/shape_predictor_68_face_landmarks.dat")

In [3]:
def calEyeArea(image):
    dets = detector(image,1)
    if(len(dets)>0):
        d = dets[0]
        shape = predictor(image,d)
        leftEye = np.empty([0,2])
        for point in [36,37,38,39,40,41]:
            leftEye = np.concatenate((leftEye,np.array([[
                shape.part(point).x,shape.part(point).y]])),
                axis=0).astype(np.int32)
            leftEyeArea = cv.contourArea(leftEye) / d.area()

        rightEye = np.empty([0,2])
        for point in [42,43,44,45,46,47]:
            rightEye = np.concatenate((rightEye,np.array([[
                shape.part(point).x,shape.part(point).y]])),
                axis=0).astype(np.int32)
            rightEyeArea = cv.contourArea(rightEye) / d.area()
        return leftEyeArea,rightEyeArea
    return -1,-1

In [4]:
openImg = cv.imread('open.jpg')
openImg = cv.cvtColor(openImg,cv.COLOR_BGR2RGB)
closedImg = cv.imread('closed.jpg')
closedImg = cv.cvtColor(closedImg,cv.COLOR_BGR2RGB)

In [5]:
leftEyeArea,rightEyeArea = calEyeArea(openImg)

In [6]:
model = tf.keras.models.load_model('./model/keras_nn/')

In [7]:
openFeature = {
    'leftEye' : np.array([[leftEyeArea]],dtype=np.float32),
    'rightEye' : np.array([[rightEyeArea]],dtype=np.float32)}

In [8]:
openPredict = model.predict(openFeature)

1/1 [==============================] - 5s 5s/step


In [ ]:
print(openPredict)
plt.figure(figsize=(6, 5))
plt.imshow(openImg)
plt.rcParams['font.sans-serif'] = ['SimHei']
open_result = "未疲劳" if openPredict[0][0] < 0.5 else "疲劳"
plt.title(f"睁眼图片\n预测值: {openPredict[0][0]:.4f}\n判断: {open_result}", fontsize=14)
plt.axis('off')
plt.show()

In [10]:
leftEyeArea,rightEyeArea = calEyeArea(closedImg)
closedPredict = model.predict({
    'leftEye' : np.array([[leftEyeArea]],dtype=np.float32),
    'rightEye' : np.array([[rightEyeArea]],dtype=np.float32)})

1/1 [==============================] - 0s 69ms/step


In [ ]:
print(closedPredict)
plt.figure(figsize=(6, 5))
plt.imshow(closedImg)
plt.rcParams['font.sans-serif'] = ['SimHei']
closed_result = "未疲劳" if closedPredict[0][0] < 0.5 else "疲劳"
plt.title(f"闭眼图片\n预测值: {closedPredict[0][0]:.4f}\n判断: {closed_result}", fontsize=14)
plt.axis('off')
plt.show()